# ఖర్చు క్లెయిమ్ విశ్లేషణ

ఈ నోట్బുക്ക് ప్రదర్శిస్తుంది ఎలా ప్లగిన్లను ఉపయోగించే ఏజెంట్లను సృష్టించగలవో, స్థానిక రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను ప్రాసెస్ చేయడానికి, ఖర్చు క్లెయిమ్ ఇమెయిల్ రూపొందించడానికి, మరియు పుంత చార్ట్ ఉపయోగించి ఖర్చు డేటాను దృశ్యీకరించడానికి. ఏజెంట్లు కార్యాచరణ నేపథ్యం ప్రకారం ఫంక్షన్లను δυναౖమిక్‌గా ఎంచుకుంటాయి.

దశలు:
1. OCR ఏజెంట్ స్థానిక రసీదు చిత్రాన్ని ప్రాసెస్ చేసి ప్రయాణ ఖర్చు సమాచారాన్ని తీయగలదు.
2. ఇమెయిల్ ఏజెంట్ ఖర్చు క్లెయిమ్ ఇమెయిల్ రూపొందిస్తుంది.

### ప్రయాణ ఖర్చు దృశ్యానికి ఉదాహరణ:
మీరు వేరే నగరంలో వ్యాపార సమావేశం కోసం ప్రయాణిస్తున్న ఉద్యోగి అని ఊహించుకోండి. మీ కంపెనీ అన్ని తగిన ప్రయాణ సంబంధ ఖర్చులను అప్పగించే విధానం కలిగి ఉంది. ఇక్కడ సాధ్యమైన ప్రయాణ ఖర్చుల విభజన ఉంది:
- రవాణా:
మీ స్థానిక నగర హోమ్ నుండి గమ్యం నగరానికి రౌండ్ ట్రిప్ కోసం విమానాదాయం.
విమానాశ్రయం వెళ్ళేందుకు మరియు తిరిగి రేస్ టాక్సీ లేదా రైడ్-హైలింగ్ సేవలు.
గమ్యం నగరంలో స్థానిక రవాణా (జనసాధారణ రవాణా, అద్దె కార్లు లేదా టాక్సీలు వంటి).

- నివాసం:
మీటింగ్ వేదికకు సమీపం లో మధ్య స్థాయి వ్యాపార హోటలులో మూడు రాత్రులు ఉండటం.

- భోజనాలు:
కంపెనీ యొక్క పర్ డియం విధానం ఆధారంగా ప్రతిరోజు అలవెన్స్ మద్యం, మధ్యాహ్న భోజనం మరియు రాత్రి భోజనానికి.

- వివిధ ఖర్చులు:
విమానాశ్రయంలో పార్కింగ్ రుసుములు.
హోటల్‌లో ఇంటర్నెట్ యాక్సెస్ ఛార్జీలు.
టిప్స్ లేదా చిన్న సేవ ఛార్జీలు.

- డాక్యుమెంటేషన్:
మీరు అన్ని రసీదులు (ఫ్లైట్స్, టాక్సీలు, హోటల్, భోజనాలు మొదలైనవి) మరియు భర్తీ చేసిన ఖర్చు నివેદనాన్ని ప్రతిపాదిస్తారు పునఃపరీక్ష కోసం.


## అవసరమైన లైబ్రరీలను దిగుమతి చేసుకోండి

నోట్‌బుక్ కోసం అవసరమైన లైబ్రరీలు మరియు మాడ్యూల్‌లను దిగుమతి చేసుకోండి.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## ఖర్చు మోడల్స్ నిర్వచించండి

వ్యక్తిగత ఖర్చుల కోసం ఒక Pydantic మోడల్ మరియు వాడుకరి ప్రశ్నను నిర్మిత ఖర్చు డేటాగా మార్చవలసిన ExpenseFormatter క్లాస్ తయారుచేయండి.

ప్రతి ఖర్చు ఈ ఫార్మాట్ లో ప్రదర్శించబడుతుంది:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## టూల్‌లు నిర్వచించడం - ఈమెయిల్ తయారు చేయడం

ఖర్చు దావాకు సమర్పించేందుకు ఈమెయిల్ రూపొందించే టూల్ ఫంక్షన్ రూపొందించండి.
- ఈ టూల్ Microsoft ఏజెంట్ ఫ్రేమ్‌వర్క్ నుండి `@tool` డెకరేటర్‌ని ఉపయోగిస్తుంది.
- ఇది ఖర్చుల మొత్తం మొత్తాన్ని లెక్కిస్తుంది మరియు వివరాలను ఈమెయిల్ శరీరంలో ఫార్మాట్ చేస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను తీసివేయడానికి సాధనం

రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను తీసివేయడానికి ఒక సాధన ఫంక్షన్‌ను సృష్టించండి.
- ఈ సాధనం Microsoft Agent Framework నుండి `@tool` డెకరేటర్‌ను ఉపయోగిస్తుంది.
- ఇది రసీదు చిత్రాన్ని చదవడంతో పాటు, దాన్ని base64 గా ఎంకోడ్ చేసి ఏజెంట్ విశ్లేషించేందుకు డేటా URIను తిరిగి ఇస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ఖర్చుల ప్రాసెసింగ్

ఏజెంట్లను నిర్వచించి, వాటిని `WorkflowBuilder` ఉపయోగించి అనుసరణ విధానంలో కలపండి.
- OCR ఏజెంట్ రసీదు చిత్రంలో నుండి నిర్మిత ఖర్చుల డేటాను `load_receipt_image` టూల్ ఉపయోగించి పొందుతుంది.
- ఇమెయిల్ ఏజెంట్ పొందిన డేటాను తీసుకుని, `generate_expense_email` టూల్ ఉపయోగించి ఒక ప్రొఫెషనల్ ఖర్చు క్లెయిమ్ ఇమెయిల్ రూపొందిస్తుంది.
- `add_edge` ఉన్న `WorkflowBuilder` అంకురితో, ఒక అనుసరణ పైప్‌లైన్‌ను సృష్టిస్తుంది: OCR ఏజెంట్ → ఇమెయిల్ ఏజెంట్.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

క్రమబద్ధ పనితీరు నిర్మించి, రసీదు చిత్రం డేటాను ప్రాసెస్ చేసి, ఖర్చు క్లెయిమ్ ఇమెయిల్‌ను రూపొందించడానికి దాన్ని నడిపించండి.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
